# Import Libraries

In [1]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
import xgboost as xgb
import ta

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score
)

from tensorflow.keras import Input, Model
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [2]:
pip install plotly

Note: you may need to restart the kernel to use updated packages.


# Seed

In [3]:
SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

tf.config.experimental.enable_op_determinism()

# Data Loading

In [4]:
df = pd.read_csv("dataset_upto_20august.csv")

print(df.head())
print(df.shape)
print(df.columns.tolist())

                   Open time      Open      High       Low     Close  \
0  2018-01-01 00:00:00+00:00  13715.65  13818.55  12750.00  13380.00   
1  2018-01-02 00:00:00+00:00  13382.16  15473.49  12890.02  14675.11   
2  2018-01-03 00:00:00+00:00  14690.00  15307.56  14150.00  14919.51   
3  2018-01-04 00:00:00+00:00  14919.51  15280.00  13918.04  15059.54   
4  2018-01-05 00:00:00+00:00  15059.56  17176.24  14600.00  16960.39   

         Volume                      Close time  Quote asset volume  \
0   8609.915844  2018-01-01 23:59:59.999000 UTC        1.147997e+08   
1  20078.092111  2018-01-02 23:59:59.999000 UTC        2.797171e+08   
2  15905.667639  2018-01-03 23:59:59.999000 UTC        2.361169e+08   
3  21329.649574  2018-01-04 23:59:59.999000 UTC        3.127816e+08   
4  23251.491125  2018-01-05 23:59:59.999000 UTC        3.693220e+08   

   Number of trades  Taker buy base asset volume  \
0            105595                  3961.938946   
1            177728                 

In [5]:
# load DXY data

dxy = pd.read_csv("DXY_clean.csv")

print(dxy.head())
print(dxy.shape)
print(dxy.columns)

   Unnamed: 0        Date        DXY
0           0  2018-01-02  91.849998
1           1  2018-01-03  92.160004
2           2  2018-01-04  91.849998
3           3  2018-01-05  91.949997
4           4  2018-01-08  92.330002
(2173, 3)
Index(['Unnamed: 0', 'Date', 'DXY'], dtype='object')


In [6]:
# Clean DXY 

dxy = dxy.drop(columns=['Unnamed: 0'])

dxy['Date'] = pd.to_datetime(dxy['Date'])

dxy = dxy[['Date', 'DXY']].sort_values('Date').reset_index(drop=True)

print(dxy.head())
print(dxy.tail())
print(dxy.isna().sum())
print("Start:", dxy['Date'].min())
print("End:", dxy['Date'].max())

        Date        DXY
0 2018-01-02  91.849998
1 2018-01-03  92.160004
2 2018-01-04  91.849998
3 2018-01-05  91.949997
4 2018-01-08  92.330002
           Date        DXY
2168 2026-08-17  99.639999
2169 2026-08-18  99.650002
2170 2026-08-19  98.830002
2171 2026-08-20  98.900002
2172 2026-08-21  98.800003
Date    0
DXY     0
dtype: int64
Start: 2018-01-02 00:00:00
End: 2026-08-21 00:00:00


# Experiment 01 : ADD DRX data 

### Merging the both data

In [7]:
print(df.columns)
print(df.head())

Index(['Open time', 'Open', 'High', 'Low', 'Close', 'Volume', 'Close time',
       'Quote asset volume', 'Number of trades', 'Taker buy base asset volume',
       'Taker buy quote asset volume', 'Ignore'],
      dtype='object')
                   Open time      Open      High       Low     Close  \
0  2018-01-01 00:00:00+00:00  13715.65  13818.55  12750.00  13380.00   
1  2018-01-02 00:00:00+00:00  13382.16  15473.49  12890.02  14675.11   
2  2018-01-03 00:00:00+00:00  14690.00  15307.56  14150.00  14919.51   
3  2018-01-04 00:00:00+00:00  14919.51  15280.00  13918.04  15059.54   
4  2018-01-05 00:00:00+00:00  15059.56  17176.24  14600.00  16960.39   

         Volume                      Close time  Quote asset volume  \
0   8609.915844  2018-01-01 23:59:59.999000 UTC        1.147997e+08   
1  20078.092111  2018-01-02 23:59:59.999000 UTC        2.797171e+08   
2  15905.667639  2018-01-03 23:59:59.999000 UTC        2.361169e+08   
3  21329.649574  2018-01-04 23:59:59.999000 UTC        

In [8]:
# preparing date column in the bitcoin_data
df['Date'] = pd.to_datetime(df['Open time'], utc=True).dt.tz_localize(None).dt.normalize()

print(df[['Open time', 'Date']].head())
print(df['Date'].dtype)

                   Open time       Date
0  2018-01-01 00:00:00+00:00 2018-01-01
1  2018-01-02 00:00:00+00:00 2018-01-02
2  2018-01-03 00:00:00+00:00 2018-01-03
3  2018-01-04 00:00:00+00:00 2018-01-04
4  2018-01-05 00:00:00+00:00 2018-01-05
datetime64[ns]


In [9]:
# merging the both data

df['Date'] = pd.to_datetime(df['Date'])

df = df.sort_values('Date').reset_index(drop=True)

df = pd.merge(
    df,
    dxy,
    on='Date',
    how='left'
)

print(df[['Date', 'Close', 'DXY']].head())
print(df[['Date', 'Close', 'DXY']].tail())

print("\nMissing values:")
print(df[['Close', 'DXY']].isna().sum())

print("\nShape:", df.shape)

        Date     Close        DXY
0 2018-01-01  13380.00        NaN
1 2018-01-02  14675.11  91.849998
2 2018-01-03  14919.51  92.160004
3 2018-01-04  15059.54  91.849998
4 2018-01-05  16960.39  91.949997
           Date     Close        DXY
3149 2026-08-16  62900.00        NaN
3150 2026-08-17  64532.10  99.639999
3151 2026-08-18  64725.42  99.650002
3152 2026-08-19  69334.79  98.830002
3153 2026-08-20  73025.15  98.900002

Missing values:
Close      0
DXY      982
dtype: int64

Shape: (3154, 14)


In [35]:
df.to_csv("bitcoin_dxy_merged.csv", index=False)

In [10]:
# DXY missing value

df['DXY'] = df['DXY'].ffill()

print("Missing DXY after ffill:")
print(df['DXY'].isna().sum())

Missing DXY after ffill:
1


In [11]:
# dropping the empty row

df = df.dropna(subset=['DXY']).reset_index(drop=True)

print("Shape:", df.shape)
print("Missing values:")
print(df[['Close', 'DXY']].isna().sum())
print(df[['Date', 'Close', 'DXY']].head())

Shape: (3153, 14)
Missing values:
Close    0
DXY      0
dtype: int64
        Date     Close        DXY
0 2018-01-02  14675.11  91.849998
1 2018-01-03  14919.51  92.160004
2 2018-01-04  15059.54  91.849998
3 2018-01-05  16960.39  91.949997
4 2018-01-06  17069.79  91.949997


# EDA with DRX

In [ ]:
# Corerlation between bitcoin_price and DRX

In [34]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=df['Date'], y=df['Close'], name='Bitcoin', line=dict(color='green')), secondary_y=False)
fig.add_trace(go.Scatter(x=df['Date'], y=df['DXY'], name='DXY', line=dict(color='red')), secondary_y=True)
fig.update_layout(title=f"Bitcoin Price vs DXY (Correlation: {df['Close'].corr(df['DXY']):.3f})", xaxis_title='Date', hovermode='x unified')
fig.update_yaxes(title_text='Bitcoin Price (USD)', secondary_y=False)
fig.update_yaxes(title_text='DXY', secondary_y=True)
fig.show()

# Feature Engineering

In [13]:
# these all features improve the performance of the model in the previous notebook experoment

In [14]:
import ta

# Calculate Relative Strength Index (RSI)
df['RSI'] = ta.momentum.RSIIndicator(df['Close'], window=14).rsi()

# Calculate Moving Average Convergence Divergence (MACD)
df['MACD'] = ta.trend.MACD(df['Close']).macd()
df['MACD_Signal'] = ta.trend.MACD(df['Close']).macd_signal()

# Calculate Bollinger Bands
bb = ta.volatility.BollingerBands(df['Close'], window=20, window_dev=2)
df['BB_Middle'] = bb.bollinger_mavg()
df['BB_Upper'] = bb.bollinger_hband()
df['BB_Lower'] = bb.bollinger_lband()
df['BB_Width'] = bb.bollinger_wband()

print("Technical indicators (RSI, MACD, Bollinger Bands) added to the DataFrame.")
print("New columns added:", ['RSI', 'MACD', 'MACD_Signal', 'BB_Middle', 'BB_Upper', 'BB_Lower', 'BB_Width'])

# Display the head of the DataFrame with new indicators
display(df.tail())

Technical indicators (RSI, MACD, Bollinger Bands) added to the DataFrame.
New columns added: ['RSI', 'MACD', 'MACD_Signal', 'BB_Middle', 'BB_Upper', 'BB_Lower', 'BB_Width']


,Open time,Open,High,Low,Close,Volume,Close time,Quote asset volume,Number of trades,Taker buy base asset volume,...,Ignore,Date,DXY,RSI,MACD,MACD_Signal,BB_Middle,BB_Upper,BB_Lower,BB_Width
3148,2026-08-16 00:00:00+00:00,63086.01,63390.00,62716.00,62900.00,4737.23135,2026-08-16 23:59:59.999000+00:00,2.986980e+08,640294,2498.03555,...,0,2026-08-16,99.669998,41.953029,-245.518292,-82.594770,63846.7015,65259.703232,62433.699768,4.426233
3149,2026-08-17 00:00:00+00:00,62900.00,64610.01,62751.10,64532.10,14255.64713,2026-08-17 23:59:59.999000+00:00,9.093661e+08,2111256,7186.91098,...,0,2026-08-17,99.639999,54.280978,-154.392920,-96.954400,63877.5565,65321.781917,62433.331083,4.521856
3150,2026-08-18 00:00:00+00:00,64532.11,65058.81,64027.85,64725.42,11789.91312,2026-08-18 23:59:59.999000+00:00,7.594053e+08,1608019,5759.24912,...,0,2026-08-18,99.650002,55.486885,-65.817397,-90.726999,63914.6135,65405.180738,62424.046262,4.664245
3151,2026-08-19 00:00:00+00:00,64725.42,70000.00,64166.00,69334.79,29054.29976,2026-08-19 23:59:59.999000+00:00,1.954769e+09,4160596,15646.86527,...,0,2026-08-19,98.830002,73.461087,372.028468,1.824094,64142.3520,66924.474997,61360.229003,8.674839
3152,2026-08-20 00:00:00+00:00,69334.78,73400.00,68902.22,73025.15,35904.79287,2026-08-20 23:59:59.999000+00:00,2.562054e+09,6249114,18144.94133,...,0,2026-08-20,98.900002,80.314652,1005.218152,202.502906,64649.2155,69358.636484,59939.794516,14.569151


In [15]:
# daily return
df['Daily_Return'] = df['Close'].pct_change()
print(df[['Close', 'Daily_Return']].head(10))

      Close  Daily_Return
0  14675.11           NaN
1  14919.51      0.016654
2  15059.54      0.009386
3  16960.39      0.126222
4  17069.79      0.006450
5  16150.03     -0.053882
6  14902.54     -0.077244
7  14400.00     -0.033722
8  14907.09      0.035215
9  13238.78     -0.111914


In [16]:
# ROC

df['ROC'] = ta.momentum.ROCIndicator(
    close=df['Close'], window=7
).roc()
# it improve the model performence so we will keep it

In [17]:
# close_lag_7

df['Close_Lag_7'] = df['Close'].shift(7)
# it also improve the model performence so we will keep it

In [18]:
# sma_7 feature
df['SMA_7'] = df['Close'].rolling(7).mean()

In [19]:
# Price sma7 ratio features
df['Price_SMA7_Ratio'] = df['Close'] / df['SMA_7']

##  Features 

In [20]:
features = [
    'Open', 'High', 'Low', 'Volume',
    'RSI', 'MACD', 'MACD_Signal',
    'BB_Middle', 'BB_Upper', 'BB_Lower', 'BB_Width',
    'Daily_Return', 'ROC',
    'Close_Lag_7', 'SMA_7', 'Price_SMA7_Ratio',
    'DXY'
]

df_model = df[features + ['Close']].dropna().copy()

print("Features:", len(features))
print("Data shape:", df_model.shape)
print("Missing values:")
print(df_model.isna().sum())

Features: 17
Data shape: (3120, 18)
Missing values:
Open                0
High                0
Low                 0
Volume              0
RSI                 0
MACD                0
MACD_Signal         0
BB_Middle           0
BB_Upper            0
BB_Lower            0
BB_Width            0
Daily_Return        0
ROC                 0
Close_Lag_7         0
SMA_7               0
Price_SMA7_Ratio    0
DXY                 0
Close               0
dtype: int64


# Train test Split

In [21]:
data = df_model[features].values
target = df_model['Close'].values.reshape(-1, 1)

n_steps_in = 30

split = int(len(data) * 0.8)

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

scaler_X.fit(data[:split])
scaler_y.fit(target[:split])

scaled_data = scaler_X.transform(data)
scaled_target = scaler_y.transform(target)

print("Data shape:", data.shape)
print("Target shape:", target.shape)
print("Split:", split)

Data shape: (3120, 17)
Target shape: (3120, 1)
Split: 2496


In [22]:
## define Sequence

In [23]:
def create_sequences(data, target, n_steps):
    X, y = [], []

    for i in range(n_steps, len(data)):
        X.append(data[i-n_steps:i])
        y.append(target[i])

    return np.array(X), np.array(y)

In [24]:
# create sequence

In [25]:
X, y = create_sequences(
    scaled_data,
    scaled_target,
    n_steps_in
)

train_size = int(len(X) * 0.8)

X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

n_features = X_train.shape[2]

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)
print("Features:", n_features)

X_train: (2472, 30, 17)
X_test : (618, 30, 17)
y_train: (2472, 1)
y_test : (618, 1)
Features: 17


# MODEL + TRAINING

In [26]:
# Build LSTM Model

input_layer = Input(shape=(n_steps_in, n_features))

lstm_layer = LSTM(
    128,
    activation='relu',
    return_sequences=False
)(input_layer)

middle_layer = Dense(64, activation='relu')(lstm_layer)

dropout = Dropout(0.2)(middle_layer)

output_layer = Dense(1, activation='linear')(dropout)

lstm_model = Model(
    inputs=input_layer,
    outputs=output_layer
)

lstm_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse'
)

lstm_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 30, 17)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │        74,752 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 83,073 (324.50 KB)

 Trainable params: 83,073 (324.50 KB)

 Non-trainable params: 0 (0.00 B)

In [27]:
# Trainig the model

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6
)

history = lstm_model.fit(
    X_train,
    y_train,
    epochs=200,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

Epoch 1/200
31/31 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - loss: 0.0048 - val_loss: 0.0011 - learning_rate: 0.0010
Epoch 2/200
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0016 - val_loss: 0.0011 - learning_rate: 0.0010
Epoch 3/200
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0012 - val_loss: 0.0019 - learning_rate: 0.0010
Epoch 4/200
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0011 - val_loss: 0.0028 - learning_rate: 0.0010
Epoch 5/200
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0010 - val_loss: 0.0025 - learning_rate: 0.0010
Epoch 6/200
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 9.6656e-04 - val_loss: 0.0013 - learning_rate: 0.0010
Epoch 7/200
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 9.2807e-04 - val_loss: 8.9277e-04 - learning_rate: 5.0000e-04
Epoch 8/200
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 8.4116e-04 - val_loss: 0.0015 - learning_rate: 5.0000e-04
Epoch 9/200
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 8.6492e-04 - val_loss: 0.0010 - lear

In [28]:
# feature extrating from lstm
feature_extractor = Model(
    inputs=lstm_model.input,
    outputs=lstm_model.layers[1].output
)

z_train = feature_extractor.predict(X_train, verbose=0)
z_test = feature_extractor.predict(X_test, verbose=0)

print("z_train:", z_train.shape)
print("z_test :", z_test.shape)

z_train: (2472, 128)
z_test : (618, 128)


## XG Boost Model

In [29]:
xgb_model = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=10,
    objective='reg:squarederror',
    random_state=42
)

xgb_model.fit(
    z_train,
    y_train.ravel()
)

print("XGBoost training completed.")

XGBoost training completed.


In [30]:
y_pred_scaled = xgb_model.predict(z_test)

y_actual = scaler_y.inverse_transform(y_test)
y_pred = scaler_y.inverse_transform(
    y_pred_scaled.reshape(-1, 1)
)

print(f"MAPE: {mean_absolute_percentage_error(y_actual, y_pred)*100:.4f}%")
print(f"RMSE: {np.sqrt(mean_squared_error(y_actual, y_pred)):.2f}")
print(f"R²: {r2_score(y_actual, y_pred):.4f}")

MAPE: 11.3615%
RMSE: 12279.43
R²: 0.5255


In [31]:
n = 60

y_actual_60 = scaler_y.inverse_transform(y_test[-n:])

y_pred_60 = scaler_y.inverse_transform(
    xgb_model.predict(z_test[-n:]).reshape(-1, 1)
)

print(
    f"MAPE: "
    f"{mean_absolute_percentage_error(y_actual_60, y_pred_60)*100:.4f}%"
)

print(
    f"RMSE: "
    f"{np.sqrt(mean_squared_error(y_actual_60, y_pred_60)):.2f}"
)

print(
    f"R²: "
    f"{r2_score(y_actual_60, y_pred_60):.4f}"
)

MAPE: 8.8800%
RMSE: 6344.35
R²: -7.4306


# RESULT OF EXP 01    

#### After adding the DRX no significant improvemnt had come 